# Chapter 3 photodiode physics lab

This notebook runs entirely in your browser through JupyterLite and its Pyodide Python kernel. The functions come from the tested `photodiode.py` engine shipped with the documentation.

The models reproduce the ideal equations in Chapter 3. They do not replace measured device data.

In [ ]:
from pathlib import Path
import sys

# JupyterLite places photodiode.py beside this notebook. The fallback also
# makes the notebook convenient when opened from a repository checkout.
try:
    import photodiode
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        module_dir = parent / "KrakenOS" / "Physics"
        if (module_dir / "photodiode.py").exists():
            sys.path.insert(0, str(module_dir))
            break
    import photodiode

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatLogSlider, FloatSlider, interact
from matplotlib.ticker import FuncFormatter

plt.style.use("seaborn-v0_8-whitegrid")

POWER_UNITS = ((30, "QW"), (27, "RW"), (24, "YW"), (21, "ZW"),
               (18, "EW"), (15, "PW"), (12, "TW"), (9, "GW"),
               (6, "MW"), (3, "kW"), (0, "W"), (-3, "mW"),
               (-6, "uW"), (-9, "nW"), (-12, "pW"))

def format_power_tick(value, _position=None):
    if value <= 0:
        return "0 W"
    log_value = np.log10(value)
    exponent, unit = next(
        ((power, label) for power, label in POWER_UNITS if log_value >= power),
        POWER_UNITS[-1],
    )
    return f"{value / 10**exponent:,.3g} {unit}"

def format_log_power(log_value):
    return format_power_tick(10**log_value)

## Equation 3.11: excess-carrier profile

Move the sliders to see how $D_e$, $\tau_e$, $\Delta n_p(0)$, and $G_L$ control the profile. The diffusion length is $L_e=\sqrt{D_e\tau_e}$.

In [ ]:
@interact(
    diffusion=FloatSlider(value=25, min=5, max=80, step=1, description="D (cm2/s)"),
    lifetime_us=FloatLogSlider(value=1, base=10, min=-2, max=2, step=0.05, description="tau (us)"),
    junction=FloatLogSlider(value=1e14, base=10, min=10, max=16, step=0.1, description="delta n(0)"),
    generation=FloatLogSlider(value=1e18, base=10, min=10, max=20, step=0.1, description="G_L"),
)
def plot_carriers(diffusion, lifetime_us, junction, generation):
    lifetime_s = lifetime_us * 1e-6
    length_um = photodiode.diffusion_length(diffusion, lifetime_s) * 1e4
    x_um = np.linspace(0, max(100, 6 * length_um), 500)
    carriers = photodiode.excess_carrier_profile(
        x_um,
        diffusion_cm2_s=diffusion,
        lifetime_s=lifetime_s,
        junction_excess_cm3=junction,
        generation_cm3_s=generation,
    )
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.semilogy(x_um, carriers, color="#7b1e24", linewidth=2.5)
    ax.set(xlabel="Distance from junction (um)", ylabel="Excess carriers (cm-3)")
    ax.set_title(f"Equation 3.11: L_e = {length_um:.1f} um")
    plt.show()

## Equations 3.14, 3.19, 3.22, and 3.27

This overview generates the current-voltage family, ideal spectral cutoff, absorption curve, and responsivity curve from the engine. Change the values in the first six lines and run the cell again.

In [ ]:
temperature_k = 300.0
ideality_factor = 1.4
generation_cm3_s = (0.0, 1e11, 3e11)
bandgap_ev = 1.12
quantum_efficiency = 0.80
absorption_cm_inv = 100.0

parameters = photodiode.PhotodiodeParameters(
    temperature_k=temperature_k,
    ideality_factor=ideality_factor,
)
voltage = np.linspace(-0.5, 0.5, 500)
wavelength = np.linspace(0.3, 1.5, 500)
depth_um = np.linspace(0, 500, 500)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for generation in generation_cm3_s:
    current = photodiode.photodiode_current_density(
        voltage, parameters=parameters, generation_cm3_s=generation
    )
    axes[0, 0].plot(voltage, current, label=f"G_L={generation:.0e}")
axes[0, 0].set(xlabel="Voltage (V)", ylabel="Current density (A/cm2)", title="Equation 3.14")
axes[0, 0].legend()

axes[0, 1].plot(
    wavelength,
    photodiode.ideal_spectral_response(wavelength, bandgap_ev),
    color="#7b1e24",
)
axes[0, 1].set(xlabel="Wavelength (um)", ylabel="Spectral response", title="Equation 3.19")

axes[1, 0].plot(
    depth_um,
    photodiode.absorption_intensity(depth_um, absorption_cm_inv),
    color="#287271",
)
axes[1, 0].set(xlabel="Depth (um)", ylabel="I / I_0", title="Equation 3.22")

axes[1, 1].plot(
    wavelength,
    photodiode.responsivity(wavelength, quantum_efficiency, bandgap_ev),
    color="#c18b2e",
)
axes[1, 1].set(xlabel="Wavelength (um)", ylabel="Responsivity (A/W)", title="Equation 3.27")

fig.tight_layout()
plt.show()

## Section 3.4.1: silicon absorption and surface reflection

For a fixed beam area, power follows the same Beer-Lambert law as intensity. The green curve assumes that all source power enters the silicon. The red curve first applies the normal-incidence air-to-silicon Fresnel loss, then applies the same bulk absorption: $P(x)=(1-R)P_0e^{-\alpha x}$.

**What the source power does and does not change.** Raising $P_0$ does not change the decay length $1/\alpha$: that belongs to the material, and because Beer-Lambert is multiplicative the *fractional* profile is identical at every power. Plotted on a fixed logarithmic axis, more power simply lifts the whole line.

What power does move is how deep the beam stays above an **absolute** level -- a detector noise floor, a damage threshold, a "fully absorbed" criterion. Inverting Equation 3.22 for depth,

$$z_{\text{floor}} = \frac{1}{\alpha}\ln\!\left(\frac{(1-R)P_0}{P_{\text{floor}}}\right),$$

so every decade of source power buys a further $\ln(10)/\alpha$ -- at $\alpha=100\ \text{cm}^{-1}$ that is 230.3 um. The depth axis follows that crossing, which is why the window widens as you turn the power up.

In [ ]:
@interact(
    incident_power_w=FloatLogSlider(value=0.1, base=10, min=-3, max=1, step=0.02, description="P0 (W)"),
    floor_power_w=FloatLogSlider(value=1e-9, base=10, min=-12, max=-3, step=0.05, description="floor (W)"),
    absorption_cm_inv=FloatLogSlider(value=100, base=10, min=2, max=5, step=0.05, description="alpha (cm-1)"),
    silicon_index=FloatSlider(value=3.5, min=3.2, max=4.2, step=0.01, description="n silicon"),
    displayed_lengths=FloatSlider(value=5, min=1, max=25, step=0.1, description="min depth (1/alpha)"),
)
def plot_silicon_absorption(
    incident_power_w,
    floor_power_w,
    absorption_cm_inv,
    silicon_index,
    displayed_lengths,
):
    absorption_length_um = 1e4 / absorption_cm_inv
    reflectance = photodiode.fresnel_reflectance(1.0, silicon_index)
    # The one depth that source power moves. See the markdown above: 1/alpha is the
    # material's, this crossing is the beam's, and it slides ln(10)/alpha per decade.
    floor_depth_um = photodiode.absorption_depth_for_power(
        floor_power_w,
        absorption_cm_inv,
        incident_power_w,
        surface_reflectance=reflectance,
    )
    gain_per_decade_um = photodiode.absorption_depth_gain_per_decade(absorption_cm_inv)
    maximum_depth_um = max(displayed_lengths * absorption_length_um, 1.05 * floor_depth_um)
    depth_um = np.linspace(0, maximum_depth_um, 600)
    no_surface_loss = photodiode.absorption_power(
        depth_um, absorption_cm_inv, incident_power_w
    )
    with_surface_loss = photodiode.absorption_power(
        depth_um,
        absorption_cm_inv,
        incident_power_w,
        surface_reflectance=reflectance,
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.semilogy(depth_um, no_surface_loss, label="no surface reflection", color="#287271", linewidth=2.5)
    ax.semilogy(depth_um, with_surface_loss, label="air-to-silicon surface", color="#7b1e24", linewidth=2.5)
    ax.axhline(floor_power_w, color="#345995", linestyle="--", linewidth=1.5, label="detection floor")
    if 0.0 < floor_depth_um <= maximum_depth_um:
        ax.axvline(floor_depth_um, color="#345995", linestyle=":", linewidth=1.2)
    ax.set(xlabel="Depth inside silicon (um)", ylabel="Optical power remaining (W)")
    # A FIXED frame, floor to the top of the P0 slider: raising the power lifts the lines
    # inside it. Autoscaling rescaled the axis by exactly the factor it was showing, so the
    # plot never changed as the slider moved.
    ax.set_ylim(floor_power_w / 10.0, 10.0)
    ax.yaxis.set_major_formatter(FuncFormatter(format_power_tick))
    ax.set_title(
        f"R = {100 * reflectance:.1f}%,  1/alpha = {absorption_length_um:.1f} um (power-independent)\n"
        f"depth to floor = {floor_depth_um:.0f} um,  +{gain_per_decade_um:.0f} um per power decade"
    )
    ax.legend(loc="upper right")
    plt.show()

## Inverse design: source power for a silicon slab

The transmitted **fraction** is $(1-R)^2e^{-\alpha d}$ and cannot be changed by source power. The desired-transmission slider instead reports the required $\alpha$ and maximum width. The desired-output slider is an absolute power, so it can be used to calculate the required source.

In [ ]:
@interact(
    width_mm=FloatSlider(value=8, min=0.1, max=10, step=0.1, description="width (mm)"),
    absorption_cm_inv=FloatLogSlider(value=100, base=10, min=-2, max=2, step=0.02, description="alpha (cm-1)"),
    silicon_index=FloatSlider(value=3.5, min=3.2, max=4.2, step=0.01, description="n silicon"),
    desired_transmission_percent=FloatSlider(value=10, min=0.1, max=50, step=0.1, description="target (%)"),
    target_output_power_w=FloatLogSlider(value=0.1, base=10, min=-12, max=0, step=0.05, description="P out (W)"),
)
def plot_slab_inverse_design(
    width_mm,
    absorption_cm_inv,
    silicon_index,
    desired_transmission_percent,
    target_output_power_w,
):
    reflectance = photodiode.fresnel_reflectance(1.0, silicon_index)
    widths_mm = np.linspace(0, width_mm, 500)
    no_surface_log_w = np.array([
        photodiode.required_source_log10_power(
            target_output_power_w, width, absorption_cm_inv, surface_count=0
        )
        for width in widths_mm
    ])
    two_surface_log_w = np.array([
        photodiode.required_source_log10_power(
            target_output_power_w,
            width,
            absorption_cm_inv,
            surface_reflectance=reflectance,
        )
        for width in widths_mm
    ])
    log_transmission = photodiode.slab_log10_transmission(
        width_mm,
        absorption_cm_inv,
        surface_reflectance=reflectance,
    )
    try:
        required_alpha = photodiode.absorption_coefficient_for_transmission(
            desired_transmission_percent / 100,
            width_mm,
            surface_reflectance=reflectance,
        )
        maximum_width_mm = required_alpha * width_mm / absorption_cm_inv
        fraction_result = (
            f"target {desired_transmission_percent:.1f}% requires alpha <= "
            f"{required_alpha:.3g} cm-1 or width <= {maximum_width_mm:.3g} mm"
        )
    except ValueError:
        fraction_result = "target fraction exceeds the two-surface transmission ceiling"

    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.semilogy(widths_mm, 10**no_surface_log_w, color="#287271", linewidth=2.5, label="no surface reflection")
    ax.semilogy(widths_mm, 10**two_surface_log_w, color="#7b1e24", linewidth=2.5, label="two uncoated surfaces")
    ax.yaxis.set_major_formatter(FuncFormatter(format_power_tick))
    ax.set(xlabel="Silicon slab width (mm)", ylabel="Required source power")
    actual_fraction = 10**log_transmission if log_transmission > -300 else 0.0
    actual_text = f"{100 * actual_fraction:.4g}%" if log_transmission >= -5 else f"effectively 0% ({-log_transmission:.2f} decades loss)"
    ax.set_title(
        f"Actual transmission: {actual_text}\n{fraction_result}\n"
        f"Source for {format_power_tick(target_output_power_w)} output: "
        f"{format_log_power(two_surface_log_w[-1])}"
    )
    ax.legend(loc="upper left")
    plt.show()

## Figure 3.10: single-layer antireflection coating

The chapter gives the ideal index and quarter-wave thickness. The engine evaluates the full wavelength-dependent interference of one lossless film.

In [ ]:
@interact(
    design_um=FloatSlider(value=1.0, min=0.45, max=1.5, step=0.01, description="lambda_0"),
    substrate_index=FloatSlider(value=3.5, min=1.5, max=4.5, step=0.01, description="n substrate"),
    film_index=FloatSlider(value=1.87, min=1.1, max=3.0, step=0.01, description="n film"),
    thickness_factor=FloatSlider(value=1.0, min=0.4, max=1.6, step=0.01, description="t / t_qw"),
)
def plot_coating(design_um, substrate_index, film_index, thickness_factor):
    wavelength_um = np.linspace(0.35, 1.75, 600)
    quarter_wave_um = design_um / (4 * film_index)
    coated_r = photodiode.single_layer_reflectance(
        wavelength_um,
        index_incident=1.0,
        index_film=film_index,
        index_substrate=substrate_index,
        thickness_um=quarter_wave_um * thickness_factor,
    )
    uncoated_r = photodiode.fresnel_reflectance(1.0, substrate_index)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(wavelength_um, 100 * (1 - coated_r), label="coated", color="#7b1e24")
    ax.axhline(100 * (1 - uncoated_r), label="uncoated", color="#c18b2e")
    ax.set(xlabel="Wavelength (um)", ylabel="Reflection-limited efficiency (%)")
    ax.legend()
    plt.show()